# Customer Segmentation — PCA, K-Means and Business Personas

**DecodeLabs Data Science Internship · Project 3**

Discover hidden customer groups in unlabeled retail data and translate them into actionable
business personas.

**Pipeline:** Scale → Compress (PCA) → Cluster (K-Means) → Translate (Personas)

**Dataset:** [Customer Personality Analysis](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis) — 2,240 customers, 29 columns.

## 1. Setup

In [23]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "raw" / "marketing_campaign.csv"

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
print("Data file found:", DATA_PATH.exists())

Data file found: True


## 2. Load the data

The separator is detected automatically. The raw file is read only and never modified.

In [24]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"{DATA_PATH} not found - see data/raw/README.md")

raw = pd.read_csv(DATA_PATH, sep=None, engine="python")
print(f"Customers: {raw.shape[0]:,} | Columns: {raw.shape[1]}")
raw.head()

Customers: 2,240 | Columns: 29


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


### 2.1 Column overview

In [25]:
pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "non_null": raw.notna().sum(),
    "missing": raw.isna().sum(),
    "unique_values": raw.nunique(),
})

,dtype,non_null,missing,unique_values
ID,int64,2240,0,2240
Year_Birth,int64,2240,0,59
Education,str,2240,0,5
Marital_Status,str,2240,0,8
Income,float64,2216,24,1974
Kidhome,int64,2240,0,3
Teenhome,int64,2240,0,3
Dt_Customer,str,2240,0,663
Recency,int64,2240,0,100
MntWines,int64,2240,0,776


## 3. Data-quality checks

### 3.1 Missing values

In [26]:
missing = raw.isna().sum()
missing = missing[missing > 0]
pd.DataFrame({"missing": missing, "percent": (missing / len(raw) * 100).round(2)})

,missing,percent
Income,24,1.07


### 3.2 Duplicates

Rows that are identical in every column except `ID` represent the same customer recorded twice.

In [27]:
print("Exact duplicate rows:          ", int(raw.duplicated().sum()))
print("Duplicates ignoring ID:        ", int(raw.drop(columns="ID").duplicated().sum()))
print("Duplicate customer IDs:        ", int(raw["ID"].duplicated().sum()))

Exact duplicate rows:           0
Duplicates ignoring ID:         182
Duplicate customer IDs:         0


### 3.3 Constant columns

In [28]:
constant_cols = [c for c in raw.columns if raw[c].nunique(dropna=False) == 1]
pd.DataFrame({"only value": raw[constant_cols].iloc[0]})

,only value
Z_CostContact,3
Z_Revenue,11


### 3.4 Categorical columns

In [29]:
for col in ["Education", "Marital_Status"]:
    print(f"{col}:\n{raw[col].value_counts().to_string()}\n")
print("Dt_Customer example values:", raw["Dt_Customer"].head(3).tolist())

Education:
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54

Marital_Status:
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2

Dt_Customer example values: ['04-09-2012', '08-03-2014', '21-08-2013']


### 3.5 Numeric ranges

In [30]:
numeric = raw.select_dtypes(include="number")
summary = numeric.describe().T
summary["skew"] = numeric.skew()
summary.round(2)

,count,mean,std,min,25%,50%,75%,max,skew
ID,2240.0,5592.16,3246.66,0.0,2828.25,5458.5,8427.75,11191.0,0.04
Year_Birth,2240.0,1968.81,11.98,1893.0,1959.00,1970.0,1977.00,1996.0,-0.35
Income,2216.0,52247.25,25173.08,1730.0,35303.00,51381.5,68522.00,666666.0,6.76
Kidhome,2240.0,0.44,0.54,0.0,0.00,0.0,1.00,2.0,0.64
Teenhome,2240.0,0.51,0.54,0.0,0.00,0.0,1.00,2.0,0.41
Recency,2240.0,49.11,28.96,0.0,24.00,49.0,74.00,99.0,-0.00
MntWines,2240.0,303.94,336.60,0.0,23.75,173.5,504.25,1493.0,1.18
MntFruits,2240.0,26.30,39.77,0.0,1.00,8.0,33.00,199.0,2.10
MntMeatProducts,2240.0,166.95,225.72,0.0,16.00,67.0,232.00,1725.0,2.08
MntFishProducts,2240.0,37.53,54.63,0.0,3.00,12.0,50.00,259.0,1.92


### 3.6 Extreme values in birth year and income

In [31]:
for col in ["Year_Birth", "Income"]:
    values = raw[col].dropna().sort_values()
    print(f"{col:<10} lowest  {values.head(5).tolist()}")
    print(f"{'':<10} highest {values.tail(5).tolist()}\n")

Year_Birth lowest  [1893, 1899, 1900, 1940, 1941]
           highest [1995, 1995, 1995, 1996, 1996]

Income     lowest  [1730.0, 2447.0, 3502.0, 4023.0, 4428.0]
           highest [157243.0, 157733.0, 160803.0, 162397.0, 666666.0]



## 4. Data cleaning

| Issue found | Rule applied | Reason |
|---|---|---|
| `Z_CostContact`, `Z_Revenue` are constant | drop both columns | carry no information |
| `Marital_Status` labels `Alone`, `Absurd`, `YOLO` | `Alone` → `Single`; drop `Absurd`, `YOLO` | `Alone` means single; the others are not valid statuses |
| Rows identical except for `ID` | keep the first occurrence | one customer counted twice would bias a cluster |
| `Income` missing for 1.07% of customers | drop those rows | under 5% missing → drop rows |
| `Year_Birth` 1893, 1899, 1900 | drop ages above 100 | impossible ages |
| `Income` of 666,666 | drop incomes above Q3 + 3 × IQR | Tukey far-out fence for implausible values |

Labels are standardised **before** duplicates are removed, so two records that differ only by
`Alone` vs `Single` are recognised as the same customer.

Age and tenure are measured from the latest enrolment date in the data (`Dt_Customer`), because
the dataset does not state its collection date.

In [32]:
df = raw.copy()
log = [("raw data", len(df))]

df = df.drop(columns=constant_cols)

df["Marital_Status"] = df["Marital_Status"].replace({"Alone": "Single"})
df = df[~df["Marital_Status"].isin(["Absurd", "YOLO"])]
log.append(("invalid marital status removed", len(df)))

df = df[~df.drop(columns="ID").duplicated()]
log.append(("duplicates (ignoring ID) removed", len(df)))

df = df.dropna(subset=["Income"])
log.append(("missing income removed", len(df)))

df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
REFERENCE_DATE = df["Dt_Customer"].max()
df["Age"] = REFERENCE_DATE.year - df["Year_Birth"]
df = df[df["Age"] <= 100]
log.append(("age over 100 removed", len(df)))

q1, q3 = df["Income"].quantile([0.25, 0.75])
income_fence = q3 + 3 * (q3 - q1)
df = df[df["Income"] <= income_fence]
log.append((f"income above {income_fence:,.0f} removed", len(df)))

df = df.reset_index(drop=True)
cleaning_log = pd.DataFrame(log, columns=["step", "customers left"])
cleaning_log["removed"] = (-cleaning_log["customers left"].diff()).fillna(0).astype(int)
print("Reference date:", REFERENCE_DATE.date())
cleaning_log

Reference date: 2014-06-29


,step,customers left,removed
0,raw data,2240,0
1,invalid marital status removed,2236,4
2,duplicates (ignoring ID) removed,2053,183
3,missing income removed,2029,24
4,age over 100 removed,2026,3
5,"income above 167,538 removed",2025,1


### 4.1 Validation after cleaning

In [33]:
pd.Series({
    "missing values": int(df.isna().sum().sum()),
    "duplicates (ignoring ID)": int(df.drop(columns="ID").duplicated().sum()),
    "age range": f"{df['Age'].min()} – {df['Age'].max()}",
    "income range": f"{df['Income'].min():,.0f} – {df['Income'].max():,.0f}",
    "marital status labels": sorted(df["Marital_Status"].unique()),
}, name="after cleaning").to_frame()

,after cleaning
missing values,0
duplicates (ignoring ID),0
age range,18 – 74
income range,"1,730 – 162,397"
marital status labels,"[Divorced, Married, Single, Together, Widow]"


## 5. Feature engineering

| Feature | Definition |
|---|---|
| `Age` | reference year − `Year_Birth` |
| `Tenure_Days` | days between enrolment and the reference date |
| `Living_With_Partner` | 1 if Married or Together, else 0 |
| `Education_Level` | Basic = 1, Graduation = 2, 2n Cycle / Master = 3, PhD = 4 |
| `Campaigns_Accepted` | accepted offers across the 5 campaigns and the last campaign (`Response`) |
| `Total_Spent` | sum of the six product spending columns |
| `Total_Purchases` | web + catalogue + store purchases |
| `Children` | `Kidhome` + `Teenhome` |

`2n Cycle` is the European second-cycle (Master's-level) degree.

In [34]:
mnt_cols = ["MntWines", "MntFruits", "MntMeatProducts",
            "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
campaign_cols = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
                 "AcceptedCmp4", "AcceptedCmp5", "Response"]

df["Tenure_Days"] = (REFERENCE_DATE - df["Dt_Customer"]).dt.days
df["Living_With_Partner"] = df["Marital_Status"].isin(["Married", "Together"]).astype(int)
df["Education_Level"] = df["Education"].map(
    {"Basic": 1, "Graduation": 2, "2n Cycle": 3, "Master": 3, "PhD": 4})
df["Campaigns_Accepted"] = df[campaign_cols].sum(axis=1)
df["Total_Spent"] = df[mnt_cols].sum(axis=1)
df["Total_Purchases"] = df[["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]].sum(axis=1)
df["Children"] = df["Kidhome"] + df["Teenhome"]

assert df["Education_Level"].notna().all(), "unmapped education label"
df[["Age", "Tenure_Days", "Living_With_Partner", "Education_Level",
    "Campaigns_Accepted", "Total_Spent", "Total_Purchases", "Children"]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Age,2025.0,45.12,11.66,18.0,37.0,44.0,55.0,74.0
Tenure_Days,2025.0,352.47,202.44,0.0,178.0,352.0,528.0,699.0
Living_With_Partner,2025.0,0.64,0.48,0.0,0.0,1.0,1.0,1.0
Education_Level,2025.0,2.67,0.84,1.0,2.0,2.0,3.0,4.0
Campaigns_Accepted,2025.0,0.45,0.89,0.0,0.0,0.0,1.0,5.0
Total_Spent,2025.0,607.71,603.46,5.0,69.0,397.0,1048.0,2525.0
Total_Purchases,2025.0,12.56,7.19,0.0,6.0,12.0,18.0,32.0
Children,2025.0,0.95,0.75,0.0,0.0,1.0,1.0,3.0


## 6. Clustering features

Twenty features describing demographics, relationship with the company, spending and
shopping channels. Totals (`Total_Spent`, `Total_Purchases`, `Children`) are excluded because they
are sums of columns already included; they are kept for persona profiling, together with
`Education`, `Marital_Status`, the individual campaign columns and `Complain`.

In [35]:
CLUSTER_FEATURES = [
    "Age", "Income", "Education_Level", "Living_With_Partner", "Kidhome", "Teenhome",
    "Tenure_Days", "Recency",
    *mnt_cols,
    "NumDealsPurchases", "NumWebPurchases", "NumCatalogPurchases",
    "NumStorePurchases", "NumWebVisitsMonth",
    "Campaigns_Accepted",
]
X = df[CLUSTER_FEATURES]

print(f"Clustering matrix: {X.shape[0]:,} customers x {X.shape[1]} features")
print("Missing values:", int(X.isna().sum().sum()))
X.describe().T[["mean", "std", "min", "max"]].round(2)

Clustering matrix: 2,025 customers x 20 features
Missing values: 0


,mean,std,min,max
Age,45.12,11.66,18.0,74.0
Income,52036.03,21603.62,1730.0,162397.0
Education_Level,2.67,0.84,1.0,4.0
Living_With_Partner,0.64,0.48,0.0,1.0
Kidhome,0.44,0.54,0.0,2.0
Teenhome,0.51,0.55,0.0,2.0
Tenure_Days,352.47,202.44,0.0,699.0
Recency,48.91,28.97,0.0,99.0
MntWines,305.48,337.64,0.0,1493.0
MntFruits,26.24,39.71,0.0,199.0
